# Aula 18 — Comparações múltiplas, FDR e ANOVA

Laboratório reproduzível da disciplina **02-statistics**.

O objetivo é observar a inflação do erro tipo I, implementar ajustes auditáveis e comparar três variantes com ANOVA, tamanho de efeito e Tukey HSD.

**Ambiente:** Python 3.10+; NumPy ≥ 1.24; pandas ≥ 2.0; SciPy ≥ 1.15; Matplotlib ≥ 3.7.  
**Seed global:** `20260907`.


## Protocolo metodológico

- A família e a taxa de erro são definidas antes da simulação.
- `alpha = q = 0,05` serve como convenção didática, não como limiar universal.
- As simulações distinguem hipóteses nulas verdadeiras e efeitos reais.
- A ANOVA usa grupos independentes simulados; dados pareados exigiriam outro modelo.
- Resultados incluem estimativas e efeito, não apenas p-values.


In [ ]:
# Dependências: numpy>=1.24, pandas>=2.0, scipy>=1.15, matplotlib>=3.7
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from scipy import stats

SEED = 20260907
ALPHA = 0.05
rng = np.random.default_rng(SEED)

print(f"Seed: {SEED}")
print(f"NumPy {np.__version__} | pandas {pd.__version__} | SciPy {scipy.__version__} | Matplotlib {matplotlib.__version__}")


## 1. Ajustes auditáveis

As funções abaixo retornam p-values ajustados na ordem original. Holm aplica máximo acumulado na ordem crescente; BH aplica mínimo acumulado na ordem reversa. Essas operações garantem monotonicidade.


In [ ]:
def ajustar_bonferroni(p_values):
    p = np.asarray(p_values, dtype=float)
    return np.minimum(p * p.size, 1.0)


def ajustar_holm(p_values):
    p = np.asarray(p_values, dtype=float)
    m = p.size
    ordem = np.argsort(p)
    p_ord = p[ordem]
    ajustados_ord = np.maximum.accumulate((m - np.arange(m)) * p_ord)
    ajustados_ord = np.minimum(ajustados_ord, 1.0)
    ajustados = np.empty(m)
    ajustados[ordem] = ajustados_ord
    return ajustados


def ajustar_bh(p_values):
    p = np.asarray(p_values, dtype=float)
    m = p.size
    ordem = np.argsort(p)
    p_ord = p[ordem]
    brutos = p_ord * m / np.arange(1, m + 1)
    ajustados_ord = np.minimum.accumulate(brutos[::-1])[::-1]
    ajustados_ord = np.minimum(ajustados_ord, 1.0)
    ajustados = np.empty(m)
    ajustados[ordem] = ajustados_ord
    return ajustados


p_teste = np.array([0.004, 0.012, 0.030, 0.040])
checagem = pd.DataFrame({
    "p_bruto": p_teste,
    "Bonferroni": ajustar_bonferroni(p_teste),
    "Holm": ajustar_holm(p_teste),
    "BH": ajustar_bh(p_teste),
})
print(checagem.round(4).to_string(index=False))

assert np.all((checagem.iloc[:, 1:] >= 0) & (checagem.iloc[:, 1:] <= 1))
assert np.all(ajustar_holm(p_teste)[np.argsort(p_teste)][:-1] <= ajustar_holm(p_teste)[np.argsort(p_teste)][1:])
assert np.all(ajustar_bh(p_teste)[np.argsort(p_teste)][:-1] <= ajustar_bh(p_teste)[np.argsort(p_teste)][1:])


### Benjamini–Hochberg passo a passo

Para `q=0,05`, encontramos o maior índice cujo p-value ordenado não ultrapassa `i*q/m`. Todos os p-values até esse índice são rejeitados.


In [ ]:
p_exemplo = np.array([0.003, 0.011, 0.018, 0.041, 0.200, 0.700])
m = len(p_exemplo)
indices = np.arange(1, m + 1)
limiares = indices * ALPHA / m
passa = p_exemplo <= limiares
k = int(indices[passa].max()) if np.any(passa) else 0

tabela_bh = pd.DataFrame({"i": indices, "p_ordenado": p_exemplo, "i*q/m": limiares, "passa": passa})
print(tabela_bh.round(4).to_string(index=False))
print(f"Maior k = {k}; hipóteses rejeitadas = {list(range(1, k + 1))}")

assert k == 3
assert np.array_equal(ajustar_bh(p_exemplo) <= ALPHA, np.arange(m) < k)


## 2. Cem testes sob a hipótese nula

Sob uma hipótese nula contínua bem calibrada, o p-value é uniforme em `[0,1]`. Simulamos 15.000 famílias de 100 testes independentes e medimos a proporção com pelo menos uma rejeição falsa.

Sob o nulo global, toda rejeição é falsa; por isso, FDR e probabilidade de qualquer rejeição coincidem neste cenário.


In [ ]:
N_EXPERIMENTOS = 15_000
M_TESTES = 100
rng_nulo = np.random.default_rng(SEED + 1)
p_nulo = rng_nulo.uniform(size=(N_EXPERIMENTOS, M_TESTES))

qualquer_bruto = np.any(p_nulo <= ALPHA, axis=1)
qualquer_bonf = np.any(p_nulo <= ALPHA / M_TESTES, axis=1)
# Sob o nulo global, Holm rejeita algo exatamente quando o menor p passa alpha/m.
qualquer_holm = qualquer_bonf.copy()

p_ord = np.sort(p_nulo, axis=1)
limiares_bh = ALPHA * np.arange(1, M_TESTES + 1) / M_TESTES
qualquer_bh = np.any(p_ord <= limiares_bh, axis=1)

taxa_teorica_sem_ajuste = 1 - (1 - ALPHA) ** M_TESTES
resultado_nulo = pd.DataFrame({
    "procedimento": ["Sem ajuste", "Bonferroni", "Holm", "BH"],
    "P(≥1 falso positivo)": [qualquer_bruto.mean(), qualquer_bonf.mean(), qualquer_holm.mean(), qualquer_bh.mean()],
})
print(resultado_nulo.round(5).to_string(index=False))
print(f"Valor teórico sem ajuste = {taxa_teorica_sem_ajuste:.6f}")

assert abs(qualquer_bruto.mean() - taxa_teorica_sem_ajuste) < 0.005
assert qualquer_bonf.mean() < 0.06
assert qualquer_bh.mean() < 0.06


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
cores = ["#dc2626", "#2563eb", "#0f766e", "#7c3aed"]
ax.bar(resultado_nulo["procedimento"], resultado_nulo["P(≥1 falso positivo)"], color=cores)
ax.axhline(ALPHA, color="#111827", linestyle="--", label="nível 0,05")
ax.set(ylim=(0, 1.05), ylabel="Proporção de famílias", title="Ao menos um falso positivo em 100 testes sob H₀")
ax.legend()
plt.tight_layout()
plt.show()


## 3. Quando alguns efeitos são reais

Geramos 100 estatísticas z: 80 hipóteses nulas verdadeiras e 20 efeitos reais. Como conhecemos a verdade da simulação, podemos contar descobertas (`R`), falsas descobertas (`V`) e poder entre os 20 efeitos.


In [ ]:
rng_mistura = np.random.default_rng(SEED + 2)
n_nulas = 80
n_efeitos = 20
mu = np.r_[np.zeros(n_nulas), np.full(n_efeitos, 3.0)]
z = rng_mistura.normal(loc=mu, scale=1.0)
p_mistura = 2 * stats.norm.sf(np.abs(z))
verdade_nula = np.arange(M_TESTES) < n_nulas

ajustes = {
    "Sem ajuste": p_mistura,
    "Bonferroni": ajustar_bonferroni(p_mistura),
    "Holm": ajustar_holm(p_mistura),
    "BH": ajustar_bh(p_mistura),
}

linhas = []
for nome, p_aj in ajustes.items():
    rejeita = p_aj <= ALPHA
    R = int(rejeita.sum())
    V = int(np.sum(rejeita & verdade_nula))
    S = int(np.sum(rejeita & ~verdade_nula))
    linhas.append({
        "procedimento": nome,
        "descobertas_R": R,
        "falsas_V": V,
        "verdadeiras_S": S,
        "FDP_observada": V / max(R, 1),
        "poder_observado": S / n_efeitos,
    })

resultado_mistura = pd.DataFrame(linhas)
print(resultado_mistura.round(3).to_string(index=False))

assert resultado_mistura["descobertas_R"].between(0, M_TESTES).all()
assert resultado_mistura["falsas_V"].le(resultado_mistura["descobertas_R"]).all()
assert resultado_mistura.loc[resultado_mistura.procedimento == "BH", "verdadeiras_S"].iloc[0] >= resultado_mistura.loc[resultado_mistura.procedimento == "Bonferroni", "verdadeiras_S"].iloc[0]


## 4. ANOVA de uma via

Simulamos uma métrica contínua para três variantes independentes de um sistema. A ANOVA clássica testa igualdade das médias; a versão de Welch relaxa a igualdade de variâncias.


In [ ]:
rng_anova = np.random.default_rng(SEED + 3)
nomes = ["A — baseline", "B — otimizada", "C — candidata"]
tamanhos = [55, 60, 58]
medias_pop = [0.62, 0.66, 0.71]
desvios_pop = [0.075, 0.080, 0.070]
grupos = [rng_anova.normal(mu_i, sd_i, n_i) for mu_i, sd_i, n_i in zip(medias_pop, desvios_pop, tamanhos)]

descritiva = pd.DataFrame({
    "grupo": nomes,
    "n": [len(g) for g in grupos],
    "média": [np.mean(g) for g in grupos],
    "desvio_padrão": [np.std(g, ddof=1) for g in grupos],
})
print(descritiva.round(4).to_string(index=False))

assert all(np.isfinite(g).all() for g in grupos)


In [ ]:
anova_classica = stats.f_oneway(*grupos, equal_var=True)
anova_welch = stats.f_oneway(*grupos, equal_var=False)

residuos = np.concatenate([g - np.mean(g) for g in grupos])
shapiro = stats.shapiro(residuos)
levene = stats.levene(*grupos, center="median")

print(f"ANOVA clássica: F = {anova_classica.statistic:.6f}; p = {anova_classica.pvalue:.9f}")
print(f"ANOVA de Welch: F = {anova_welch.statistic:.6f}; p = {anova_welch.pvalue:.9f}")
print(f"Shapiro dos resíduos: W = {shapiro.statistic:.6f}; p = {shapiro.pvalue:.6f}")
print(f"Levene (centro=mediana): W = {levene.statistic:.6f}; p = {levene.pvalue:.6f}")

assert anova_classica.pvalue < ALPHA
assert anova_welch.pvalue < ALPHA
assert levene.pvalue > 0.01  # não é prova de igualdade; apenas checagem desta simulação


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8))
bp = ax.boxplot(grupos, tick_labels=nomes, patch_artist=True, showmeans=True)
for caixa, cor in zip(bp["boxes"], ["#93c5fd", "#5eead4", "#fcd34d"]):
    caixa.set_facecolor(cor)
ax.set(ylabel="Métrica simulada", title="Distribuições observadas por variante")
plt.tight_layout()
plt.show()


### Tamanho de efeito

Calculamos as somas de quadrados diretamente. `eta²` descreve a fração da variabilidade amostral associada aos grupos; `omega²` corrige parte do viés.


In [ ]:
todos = np.concatenate(grupos)
media_geral = todos.mean()
k_grupos = len(grupos)
N = len(todos)

ss_entre = sum(len(g) * (g.mean() - media_geral) ** 2 for g in grupos)
ss_dentro = sum(np.sum((g - g.mean()) ** 2) for g in grupos)
ss_total = np.sum((todos - media_geral) ** 2)
ms_dentro = ss_dentro / (N - k_grupos)

eta2 = ss_entre / ss_total
omega2 = (ss_entre - (k_grupos - 1) * ms_dentro) / (ss_total + ms_dentro)
f_manual = (ss_entre / (k_grupos - 1)) / ms_dentro

print(f"SS_entre = {ss_entre:.6f}; SS_dentro = {ss_dentro:.6f}; SS_total = {ss_total:.6f}")
print(f"eta² = {eta2:.6f}; omega² = {omega2:.6f}")
print(f"F manual = {f_manual:.6f}")

assert np.isclose(ss_total, ss_entre + ss_dentro)
assert np.isclose(f_manual, anova_classica.statistic)
assert 0 <= omega2 <= eta2 <= 1


## 5. Pós-teste de Tukey

Como o teste global rejeitou a igualdade das médias e as premissas são razoáveis nesta simulação, usamos Tukey HSD para todas as comparações pareadas com controle de FWER.


In [ ]:
tukey = stats.tukey_hsd(*grupos)
ic_tukey = tukey.confidence_interval(confidence_level=0.95)
pares = []
for i in range(k_grupos):
    for j in range(i + 1, k_grupos):
        pares.append({
            "comparação": f"{nomes[i]} − {nomes[j]}",
            "diferença_médias": grupos[i].mean() - grupos[j].mean(),
            "p_ajustado_Tukey": tukey.pvalue[i, j],
            "IC_inferior": ic_tukey.low[i, j],
            "IC_superior": ic_tukey.high[i, j],
        })

tabela_tukey = pd.DataFrame(pares)
tabela_tukey_exibicao = tabela_tukey.copy()
tabela_tukey_exibicao["p_ajustado_Tukey"] = tabela_tukey_exibicao["p_ajustado_Tukey"].map(lambda x: f"{x:.3e}")
for coluna in ["diferença_médias", "IC_inferior", "IC_superior"]:
    tabela_tukey_exibicao[coluna] = tabela_tukey_exibicao[coluna].map(lambda x: f"{x:.6f}")
print(tabela_tukey_exibicao.to_string(index=False))

assert len(tabela_tukey) == 3
assert tabela_tukey["p_ajustado_Tukey"].between(0, 1).all()
assert (tabela_tukey["IC_inferior"] <= tabela_tukey["diferença_médias"]).all()
assert (tabela_tukey["diferença_médias"] <= tabela_tukey["IC_superior"]).all()


## 6. Checagens finais

As asserções consolidam as relações que devem ser verdadeiras independentemente da interpretação substantiva. Execute o notebook de cima para baixo.


In [ ]:
assert resultado_nulo.loc[0, "P(≥1 falso positivo)"] > 0.98
assert resultado_nulo.loc[1, "P(≥1 falso positivo)"] < 0.06
assert k == 3
assert anova_classica.pvalue < ALPHA
assert np.isclose(f_manual, anova_classica.statistic)
assert len(tabela_tukey) == k_grupos * (k_grupos - 1) // 2

print("Todas as verificações foram aprovadas.")
print(f"FWER sem ajuste em 100 testes: {qualquer_bruto.mean():.6f}")
print(f"FWER Bonferroni: {qualquer_bonf.mean():.6f} | rejeição global BH: {qualquer_bh.mean():.6f}")
print(f"ANOVA: F={anova_classica.statistic:.6f}, p={anova_classica.pvalue:.9f}, eta²={eta2:.6f}, omega²={omega2:.6f}")


## Desafios

1. Troque `M_TESTES` por 20 e confirme a probabilidade teórica `1 - 0,95**20`.
2. Varie a média dos efeitos reais e compare o poder de Holm e BH.
3. Crie correlação entre os testes e discuta se as garantias do BH clássico ainda são adequadas.
4. Aumente a variância do grupo C e compare ANOVA clássica e de Welch.
5. Simule resultados pareados por seed e explique por que este código de ANOVA independente não deve ser reutilizado sem adaptação.
